In [2]:
from dotenv import load_dotenv
load_dotenv()

from langgraph.graph import START, END, StateGraph
from langchain_groq import ChatGroq
from pydantic import BaseModel
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from typing import Literal

llm = ChatGroq(
    model="openai/gpt-oss-20b",
)

class FlowState(BaseModel):
    query: str = ""
    draft: str = ""
    human_fd: str = ""
    final_res: str = ""


def draftmail_node(state: FlowState) -> FlowState:
    if state.human_fd:
        prompt = f"""
Rewrite this email draft based on the human feedback.

Query: {state.query}

Original draft:
{state.draft}

Human feedback:
{state.human_fd}
"""
        res = llm.invoke(prompt).content
        state.human_fd = ""
    else:
        res = llm.invoke(state.query).content

    state.draft = res
    return state


def human_feedback_node(state: FlowState) -> FlowState:
    fd = interrupt({
        "draft_mail": state.draft,
        "question": "Do you want to continue or re-write the mail?"
    })

    feedback = str(fd or "").strip().lower()

    if feedback in {"ok", "okay", "continue", "approved", "done", "yes"}:
        state.human_fd = ""
    else:
        state.human_fd = feedback

    return state


def final_node(state: FlowState) -> FlowState:
    state.final_res = state.draft
    print("mail sent successfully...")
    return state


def conditional_node(state: FlowState) -> Literal["draftmail_node", "final_node"]:
    if state.human_fd:
        return "draftmail_node"
    return "final_node"


graph = StateGraph(FlowState)
graph.add_node("draftmail_node", draftmail_node)
graph.add_node("human_feedback_node", human_feedback_node)
graph.add_node("final_node", final_node)

graph.add_edge(START, "draftmail_node")
graph.add_edge("draftmail_node", "human_feedback_node")
graph.add_conditional_edges("human_feedback_node", conditional_node)
graph.add_edge("final_node", END)

memory = InMemorySaver()
graph = graph.compile(checkpointer=memory)

In [3]:
thread_config = {
    "configurable": {
        "thread_id": "loop-1"
    }
}

res = graph.invoke(
    {"query": "Write an email to abc@gmail.com about sick leave for 3 days."},
    config=thread_config
)

res

{'query': 'Write an email to abc@gmail.com about sick leave for 3 days.',
 'draft': '**Subject:** Request for Sick Leave (3 Days)\n\nDear [Supervisor’s Name],\n\nI hope you are well. I am writing to inform you that I am unwell and, following my doctor’s advice, will need to take sick leave for the next three days.\n\n**Leave dates:**  \n- **Start:** [Start Date]  \n- **End:** [End Date]  \n\nDuring this period, I will be unable to attend work or respond promptly to emails. I will keep you updated on my recovery and will return to work on [Return Date], provided my health permits.\n\nIf there are any urgent matters that require attention, please let me know, and I will do my best to assist remotely or delegate as needed.\n\nThank you for your understanding and support.\n\nKind regards,\n\n[Your Full Name]  \n[Your Position]  \n[Your Contact Number]  \n[Your Email Address]',
 'human_fd': '',
 'final_res': '',
 '__interrupt__': [Interrupt(value={'draft_mail': '**Subject:** Request for Sic

In [4]:
res = graph.invoke(
    Command(resume="No, please re-write"),
    config=thread_config
)

res

{'query': 'Write an email to abc@gmail.com about sick leave for 3 days.',
 'draft': '**Subject:** Sick Leave Request – 3 Days (March\u202f1–3,\u202f2024)\n\nHi [Supervisor’s Name],\n\nI’m writing to let you know that I’m unwell and have been advised by my doctor to take a short break to recover. I will need to be on sick leave for the next three days.\n\n- **Leave period:** March\u202f1\u202f–\u202fMarch\u202f3,\u202f2024  \n- **Expected return:** March\u202f4,\u202f2024 (subject to my health)\n\nDuring this time I will not be able to attend the office or respond quickly to emails. If anything urgent comes up, please feel free to email me or call me at [Your Phone Number], and I’ll do my best to assist remotely or arrange for a colleague to cover.\n\nThank you for your understanding.\n\nBest regards,\n\n[Your Full Name]  \n[Your Position]  \n[Your Phone Number]  \n[Your Email Address]',
 'human_fd': '',
 'final_res': '',
 '__interrupt__': [Interrupt(value={'draft_mail': '**Subject:** S

In [5]:
res = graph.invoke(
    Command(resume="yes"),
    config=thread_config
)

res

mail sent successfully...


{'query': 'Write an email to abc@gmail.com about sick leave for 3 days.',
 'draft': '**Subject:** Sick Leave Request – 3 Days (March\u202f1–3,\u202f2024)\n\nHi [Supervisor’s Name],\n\nI’m writing to let you know that I’m unwell and have been advised by my doctor to take a short break to recover. I will need to be on sick leave for the next three days.\n\n- **Leave period:** March\u202f1\u202f–\u202fMarch\u202f3,\u202f2024  \n- **Expected return:** March\u202f4,\u202f2024 (subject to my health)\n\nDuring this time I will not be able to attend the office or respond quickly to emails. If anything urgent comes up, please feel free to email me or call me at [Your Phone Number], and I’ll do my best to assist remotely or arrange for a colleague to cover.\n\nThank you for your understanding.\n\nBest regards,\n\n[Your Full Name]  \n[Your Position]  \n[Your Phone Number]  \n[Your Email Address]',
 'human_fd': '',
 'final_res': '**Subject:** Sick Leave Request – 3 Days (March\u202f1–3,\u202f2024)